# Lab 2 Convolutional Neural Networks

This lab has two parts. In **Part A**, we will go through the basics of CNN and learn two architectures, AlexNet and ResNet. You will have an assignment for this lab, in group of 3. You may use **Part B** as a starting point to implement those required architectures and complete the assignment tasks.

You will need to submit your code and a written report as a group to address all tasks described in the assignment description.

# Part A Introduction and demonstration
We will build a small CNN for CIFAR-10 and follow the complete prediction workflow, including some explorations of the dataset, a simple network, training, evaluation and some basic demonstrations of the tasks that you will explore in the assignment.

## A1 Libraries and data
NumPy and pandas manage data and perform computations, Matplotlib creates visualizations, torchvision loads images and applies transforms, and PyTorch defines and trains the networks. We import each tool before using it.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms
from torchvision.transforms import functional as TF
from pathlib import Path
# cpu as a fallback
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

**Load the training split.** [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) has 50,000 training and 10,000 test RGB images in ten classes. The images are 32×32 pixels. We will split training data for development and leave the official test set for the end.

In [ ]:
from torchvision.datasets.utils import download_and_extract_archive
DATA_ROOT = Path("data")
try:
  database = datasets.CIFAR10(DATA_ROOT, train=True, download=False)
except:
  # faster than download with datasets
  download_and_extract_archive(
      "https://data.brainchip.com/dataset-mirror/cifar10/cifar-10-python.tar.gz",
      download_root=str(DATA_ROOT),
      filename="cifar-10-python.tar.gz",
      md5="c58f30108f718f92721af3b95e74349a",
  )
  database = datasets.CIFAR10(DATA_ROOT, train=True, download=False)

labels = np.asarray(database.targets)
class_names = database.classes

print("Images:", database.data.shape, database.data.dtype)
print("Pixel range:", database.data.min(), database.data.max())
display(
    pd.DataFrame(
        {
            "class": class_names,
            "count": np.bincount(labels, minlength=10),
        }
    )
)

**Observe the data.** The stored array uses [N,H,W,C]. A convolution uses [B,C,H,W].

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))
for target, ax in enumerate(axes.flat):
    image_id = int(np.flatnonzero(labels == target)[0])
    ax.imshow(database.data[image_id])
    ax.set_title(f"{class_names[target]} · ID {image_id}")
    ax.axis("off")
fig.tight_layout()
plt.show()

## A2 Prepare the data
**Train/validation/test split.** A fixed, stratified split keeps the comparison reproducible. Training set fits parameters, validation set supports design decisions, and the test set evaluates the frozen model.

In [ ]:
from sklearn.model_selection import train_test_split

SPLIT_SEED = 2223
train_ids, val_ids = train_test_split(
    np.arange(len(database)),
    test_size=0.1,
    stratify=labels,
    random_state=SPLIT_SEED,
)
train_ids, val_ids = np.sort(train_ids), np.sort(val_ids)
assert set(train_ids).isdisjoint(val_ids)
print("Training / validation:", len(train_ids), len(val_ids))

**Fit normalization on training images only.** For each channel $c$ and $M$ training images,

$$\mu_c=\frac1M\sum_i x_{ic},\qquad
\sigma_c=\sqrt{\frac1M\sum_i x_{ic}^2-\mu_c^2},\qquad
\widehat{x}_{ic}=\frac{x_{ic}-\mu_c}{\sigma_c}.$$

In [ ]:
from tqdm import tqdm

channel_sum = np.zeros(3)
channel_square_sum = np.zeros(3)
pixel_count = 0

# we process images in a batch of 256
for start in tqdm(range(0, len(train_ids), 256)):
    pixels = database.data[train_ids[start : start + 256]].astype(np.float64) / 255
    channel_sum += pixels.sum(axis=(0, 1, 2)) # sum across samples in a batch, image heights, image widths
    channel_square_sum += np.square(pixels).sum(axis=(0, 1, 2))
    pixel_count += np.prod(pixels.shape[:3])

mean = channel_sum / pixel_count
std = np.sqrt(np.maximum(channel_square_sum / pixel_count - mean**2, 1e-12))
stats = {"mean": mean.tolist(), "std": std.tolist()}
display(pd.DataFrame(stats, index=["red", "green", "blue"]))

**Save the raw image ids.** Our small CNN uses the native 32×32 images. A dataset view returns a raw [0,1] tensor, label and original image ID.

In [ ]:
class RawImages(Dataset):
    def __init__(self, dataset, ids):
        self.dataset = dataset
        self.ids = list(map(int, ids))

    def __len__(self):
        return len(self.ids)

    def __getitem__(self, idx):
        image_id = self.ids[idx]
        image, target = self.dataset[image_id]
        return TF.to_tensor(image), int(target), image_id

**Prepare a batch.** Normalize using training statistics. The demo keeps `size = 32`. Part B will use an explicitly chosen input resolution for each architecture. Keeping this differentiable operation inside the model call also allows later gradients with respect to pixels.

In [ ]:
def prepare_inputs(raw, stats, size):
    resized = nn.functional.interpolate(
        raw,
        size=(size, size),
        mode="bilinear",
        align_corners=False,
        antialias=True,
    )
    mean = resized.new_tensor(stats["mean"])[None, :, None, None]
    std = resized.new_tensor(stats["std"])[None, :, None, None]
    return (resized - mean) / std

## A3 Conv2d
`nn.Conv2d` applies a 2D convolution over an input signal of size $(B, C_{in}, H_{in}, W_{in})$ and gives output of size $(B, C_{out}, H_{out}, W_{out})$, where
$$
H_{out} = \left\lfloor\frac{H_{in}+2\times \text{padding}[0] - \text{dilation}[0]\times (\text{kernel_size}[0]-1)-1}{\text{stride}[0]} + 1\right\rfloor,
$$\
$$
W_{out} = \left\lfloor\frac{W_{in}+2\times \text{padding}[1] - \text{dilation}[1]\times (\text{kernel_size}[1]-1)-1}{\text{stride}[1]} + 1\right\rfloor.
$$
From documentation:


> `class torch.nn.Conv2d(in_channels, out_channels, kernel_size, stride=1, padding=0, dilation=1, groups=1, bias=True, padding_mode='zeros', device=None, dtype=None)`

**Verify one layer.**

In [ ]:
probe_layer = nn.Conv2d(3, 16, kernel_size=3, padding=1)
with torch.no_grad():
    probe_output = probe_layer(torch.zeros(2, 3, 32, 32))
print("Output:", probe_output.shape)

## A4 A small CNN classifier
We use a small network with two convolutional stages and a dense classification head as demonstrative example. ReLU adds nonlinearity, max pooling reduces resolution, and global average pooling produces one feature per channel. For the default shape,

$$[B,3,32,32]\to[B,16,16,16]\to[B,32,8,8]\to[B,32]\to[B,10].$$

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self, n_channels=16, dropout=0.0):
        super().__init__()
        self.network = nn.Sequential(
            nn.Conv2d(3, n_channels, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(n_channels, 2 * n_channels, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.head = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(dropout),
            nn.Linear(2 * n_channels, 10),
        )

    def forward(self, images):
        return self.head(self.network(images))

**Model factory.** We can add a `build_model` function as a model factory to build model from configs more easily.

In [ ]:
def build_model(name, options=None):
    options = {} if options is None else dict(options)
    if name != "small_cnn":
        raise ValueError(name)
    return SmallCNN(**options).to(device), 32

**(Optional) Trace actual stage shapes.** You can add forward hooks to record module outputs.

In [ ]:
@torch.inference_mode()
def trace_shapes(model, size):
    rows, hooks = [], []

    def record(name):
        def hook(layer, inputs, output):
            rows.append({"stage": name, "shape": tuple(output.shape)})
        return hook

    for name, layer in model.named_children():
        hooks.append(layer.register_forward_hook(record(name)))
    model.eval()
    logits = model(torch.zeros(1, 3, size, size, device=device))
    for hook in hooks:
        hook.remove()
    assert logits.shape == (1, 10) and torch.isfinite(logits).all()
    return rows

**Check a forward pass.** Check the model architecture and trace the output shapes for each stage.

In [ ]:
demo_probe, demo_size = build_model("small_cnn")
display(trace_shapes(demo_probe, demo_size))
print("Parameters:", sum(p.numel() for p in demo_probe.parameters()))
print(demo_probe)
del demo_probe

## A5 Configure training
**Reproducible run with a seed, data order and explicit settings.** For demonstration only.

In [ ]:
import random
from dataclasses import dataclass, asdict, field
import time, json, uuid

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

@dataclass
class Run:
    name: str
    model: str = "small_cnn"
    options: dict = field(default_factory=dict)
    seed: int = 42
    epochs: int = 1
    seconds: float = 60.0
    batch_size: int = 8
    lr: float = 0.01
    l2: float = 0.0
    augmentation: bool = False

**Dataloaders.** You may configure augmentations in Part B.

In [ ]:
def make_loaders(run, training_ids, validation_ids):
    return {
        "train": DataLoader(
            RawImages(database, training_ids),
            batch_size=run.batch_size,
            shuffle=True,
            generator=torch.Generator().manual_seed(run.seed),
        ),
        "train_eval": DataLoader(
            RawImages(database, training_ids),
            batch_size=run.batch_size
        ),
        "val": DataLoader(
            RawImages(database, validation_ids),
            batch_size=run.batch_size
        ),
    }

## A6 Train and validate
Cross-entropy compares raw logits with integer class labels:
$$L_{CE}=-\frac1B\sum_i\log\frac{e^{z_{i,y_i}}}{\sum_c e^{z_{i,c}}},$$
where $e_{z_i, y_i}$ is the logit of the $i$-th sample for its correct label $y_i$.

We first train with this objective alone. Regularized objectives are introduced briefly after evaluation, and you will make comparisons in Part B.

**Train a batch.** Move inputs and targets to device, clear gradients, compute the loss, backpropagate and update.

In [ ]:
def train_batch(model, batch, optimizer, run, size):
    raw, targets, _ = batch
    raw, targets = raw.to(device), targets.to(device)
    optimizer.zero_grad(set_to_none=True)
    logits = model(prepare_inputs(raw, stats, size))
    ce = nn.functional.cross_entropy(logits, targets)
    ce.backward()
    optimizer.step()
    count = len(targets)
    correct = (logits.argmax(1) == targets).sum().item()
    return ce.item() * count, ce.item() * count, correct, count

**Train one epoch.**

In [ ]:
def train_epoch(model, loader, optimizer, run, size, epoch):
    model.train()
    totals = np.zeros(4)
    for batch in tqdm(loader, desc=f"[{epoch}/{run.epochs}] {run.name}"):
        totals += train_batch(model, batch, optimizer, run, size)
    ce, objective, correct, count = totals
    return {
        "ce": ce / count,
        "objective": objective / count,
        "accuracy": correct / count,
        "steps": len(loader),
    }

**Define evaluation function.**

In [ ]:
@torch.inference_mode()
def evaluate(model, loader, size):
    model.eval()
    loss_sum = 0.0
    all_ids, all_targets, all_probs = [], [], []
    for raw, targets, ids in loader:
        raw, targets = raw.to(device), targets.to(device)
        logits = model(prepare_inputs(raw, stats, size))
        loss_sum += nn.functional.cross_entropy(logits, targets, reduction="sum").item()
        all_ids.extend(ids.tolist())
        all_targets.extend(targets.cpu().tolist())
        all_probs.extend(logits.softmax(1).cpu().tolist())
    probabilities = np.asarray(all_probs)
    targets = np.asarray(all_targets)
    predictions = probabilities.argmax(1)
    return {
        "ce": loss_sum / len(targets),
        "accuracy": float(np.mean(predictions == targets)),
        "ids": np.asarray(all_ids),
        "targets": targets,
        "predictions": predictions,
        "probabilities": probabilities,
    }

**Demonstration on a small subset.**

In [ ]:
def small_subset(ids, size):
    chosen, _ = train_test_split(
        ids, train_size=size, stratify=labels[ids], random_state=SPLIT_SEED
    )
    return np.sort(chosen)

demo_train_ids = small_subset(train_ids, 5000)
demo_val_ids = small_subset(val_ids, 500)
demo_run = Run("demonstration",
               model="small_cnn",
               epochs=10,
               batch_size=32,
               lr=0.02,
               options={"n_channels": 32})
set_seed(demo_run.seed)
demo_model, demo_size = build_model(demo_run.model)
demo_loaders = make_loaders(demo_run, demo_train_ids, demo_val_ids)
demo_optimizer = torch.optim.SGD(demo_model.parameters(), lr=demo_run.lr)
demo_history = []

**Run the basic loop.** Each epoch updates weights and then evaluates validation images without gradients.

In [ ]:
for epoch in range(1, demo_run.epochs + 1):
    training = train_epoch(demo_model, demo_loaders["train"], demo_optimizer, demo_run, demo_size, epoch)
    validation = evaluate(demo_model, demo_loaders["val"], demo_size)
    demo_history.append({
        "epoch": epoch,
        "train_ce": training["ce"],
        "train_accuracy": training["accuracy"],
        "val_ce": validation["ce"],
        "val_accuracy": validation["accuracy"],
    })
display(pd.DataFrame(demo_history))

## A7 Diagnose and use the model
**Plot learning curves.** The SmallCNN network is only for demonstration and does not necessarily perform well.

In [ ]:
def plot_history(history):
    table = pd.DataFrame(history)
    fig, axes = plt.subplots(1, 2, figsize=(10, 3))
    axes[0].plot(table.epoch, table.train_ce, marker="o", label="Training")
    axes[0].plot(table.epoch, table.val_ce, marker="o", label="Validation")
    axes[1].plot(table.epoch, table.train_accuracy, marker="o", label="Training")
    axes[1].plot(table.epoch, table.val_accuracy, marker="o", label="Validation")
    for ax, ylabel in zip(axes, ("Cross-entropy", "Accuracy")):
        ax.set(xlabel="Epoch", ylabel=ylabel)
        ax.legend()
    fig.tight_layout()
    plt.show()


plot_history(demo_history)

**Predict raw images.** We use softmax to transform from model's output logits to probabilities.

In [ ]:
@torch.inference_mode()
def predict(model, raw, size):
    model.eval()
    return model(prepare_inputs(raw.to(device), stats, size)).softmax(1).cpu()


demo_result = evaluate(demo_model, demo_loaders["val"], demo_size)
raw, target, image_id = RawImages(database, demo_val_ids)[0]
probabilities = predict(demo_model, raw.unsqueeze(0), demo_size)
print("ID:", image_id, "true:", class_names[target])
print("Predicted:", class_names[int(probabilities.argmax())])

**Inspect a confusion matrix.** Aggregate accuracy can hide class-specific errors. This is validation evidence and can be used during development.

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

fig, ax = plt.subplots(figsize=(7, 6))
ConfusionMatrixDisplay.from_predictions(
    demo_result["targets"],
    demo_result["predictions"],
    labels=list(range(10)),
    display_labels=class_names,
    xticks_rotation=45,
    colorbar=False,
    ax=ax,
)
fig.tight_layout()
plt.show()

## A8 Basic extensions to the workflow

Next we will see some extensions related to the tasks you will be doing in Part B.

**Capacity and regularization.** For CNNs, capacity is influenced by:


*   Depth. Number of convolutional layers.
*   Width. Number of channels (width) per layer.
*   Kernel size. Larger kernels can capture broader spatial patterns.
*   Architecture. Residual connections, attention mechanisms, etc.

Regularization can reduce overfitting by constraining or modifying how the model learns. Common techniques include L2 loss (penalizes large weights), dropout (randomly disables activations during training) and data augmentation (introduces variations of training images). The L2 regularization is given by
$$L=L_{CE}+\frac{\lambda}{2}\sum_W\| W\|_F^2,$$
where $L_{CE}$ is the cross-entropy loss, $W$ is the weight matrix, $\|\cdot\|_F^2$ denotes the squared Forbenius norm, and $\lambda$ is the regularization strength. For a weight matrix,
$$
\|W\|_F^2 = \sum{i,j}W_{ij}^2.
$$
For convolutional kernels, the sum extends over all kernel dimensions. It is typically applied to the trainable weights of convolutional and fully connected layers.

In [ ]:
print("Narrow parameters:", sum(p.numel() for p in SmallCNN(n_channels=8).parameters()))
print("Wide parameters:", sum(p.numel() for p in SmallCNN(n_channels=32).parameters()))
dropout_demo = nn.Dropout(p=0.5)
features_demo = torch.ones(1, 12)
dropout_demo.train()
print("Training dropout:", dropout_demo(features_demo))
dropout_demo.eval()
print("Evaluation dropout:", dropout_demo(features_demo))
weights_demo = next(demo_model.parameters())
print("One weight tensor's squared norm:", weights_demo.square().sum().item())

**Augmentation and corruption.** We demonstrate random flip and darkening here. You will explore other augmentations and corruptions in the tasks.

In [ ]:
raw_demo, target_demo, id_demo = RawImages(database, demo_val_ids)[0]
flipped_demo = TF.hflip(raw_demo)
dark_demo = (0.7 * raw_demo).clamp(0, 1)
fig, axes = plt.subplots(1, 3, figsize=(8, 3))
for ax, image, title in zip(
    axes, [raw_demo, flipped_demo, dark_demo], ["Original", "Flipped", "Darkened"]
):
    ax.imshow(image.permute(1, 2, 0))
    ax.set_title(title)
    ax.axis("off")
plt.show()
print("Original and darkened predictions:")
print(predict(demo_model, torch.stack([raw_demo, dark_demo]), demo_size).argmax(1))

**One pixel-gradient adversarial.** A one-step Fast Gradient Sign Method (FGSM) perturbation increases the labelled loss within a raw-pixel bound.
$$x_{adv}=\operatorname{clip}_{[0,1]}(x+\epsilon\operatorname{sign}(\nabla_x L))$$
where:
- $x$: Original image, with pixel values in [0,1].
- $L$: Classification loss, typically cross-entropy.
- $\nabla_x L$: Gradient of loss with respect to the input pixels, not model weights.
- $\operatorname{sign}$: Converts each gradient element to -1, 0, or +1.
- $\epsilon$: Maximum permitted change to each pixel.
- $\operatorname{clip}_{[0,1]}$: Ensures resulting pixel values remain valid.

In [ ]:
demo_model.eval()
attack_input = raw_demo.unsqueeze(0).to(device).requires_grad_(True)
attack_target = torch.tensor([target_demo], device=device)
attack_logits = demo_model(prepare_inputs(attack_input, stats, demo_size))
attack_loss = nn.functional.cross_entropy(attack_logits, attack_target)
(pixel_gradient,) = torch.autograd.grad(attack_loss, attack_input)
epsilon_demo = 4 / 255
perturbed_demo = (attack_input + epsilon_demo * pixel_gradient.sign()).clamp(0, 1).detach()
print("Maximum pixel change:", (perturbed_demo - attack_input.detach()).abs().max().item())
print("Clean and perturbed predictions:")
print(predict(demo_model, torch.cat([attack_input.detach(), perturbed_demo]), demo_size).argmax(1))

**Inference timing measurement.** GPU operations are asynchronous, so synchronize around a measurement. This basic example times prepared-input model forwards. Part B adds repeated measurements, batch-size comparisons and a stated timing protocol.

In [ ]:
def synchronize():
    if device.type == "cuda":
        torch.cuda.synchronize(device)

timing_inputs = prepare_inputs(raw_demo.unsqueeze(0).to(device), stats, demo_size)
with torch.inference_mode():
    for _ in range(5):
        demo_model(timing_inputs)
    synchronize()
    timing_started = time.perf_counter()
    for _ in range(20):
        demo_model(timing_inputs)
    synchronize()
print("Mean model-only milliseconds:", (time.perf_counter() - timing_started) * 1000 / 20)

## A9 Architectures for the assignment
In the lab assignment, you will implement two CNN architectures, AlexNet and ResNet. You should implement the architectures yourself using PyTorch layers such as `nn.Conv2d`, `nn.BatchNorm2d`, `nn.ReLU`, `nn.MaxPool2d`, and `nn.Linear`. Do not use pretrained or ready-made model implementations from `torchvision.models`.

### AlexNet

AlexNet was introduced by Krizhevsky et al. (2012). It consists of five convolutional layers followed by three fully connected layers. The original architecture uses ReLU activation functions, max pooling, and dropout.

Reference: [ImageNet Classification with Deep Convolutional Neural Networks](https://proceedings.neurips.cc/paper/2012/hash/c399862d3b9d6b76c8436e924a68c45b-Abstract.html)

#### Architecture (adapted)

AlexNet follows a sequential architecture:

**Input → Conv1 → Pool → Conv2 → Pool → Conv3 → Conv4 → Conv5 → Pool → Flatten → FC1 → FC2 → FC3**

The original AlexNet was designed for approximately $224 \times 224$ RGB images. Its first convolution uses an $11 \times 11$ kernel with stride 4, which reduces the spatial dimensions substantially.

For CIFAR-10, the input images are only $32 \times 32$. Using the original configuration would reduce the spatial dimensions too quickly. We therefore use smaller kernels and adjust the pooling operations.

#### Suggested CIFAR-10 adaptation

| Layer | Configuration | Output shape (C × H × W) |
|---|---|---|
| Input | RGB image | 3 × 32 × 32 |
| Conv1 | 64 filters, 3×3, stride 1, padding 1 + ReLU | 64 × 32 × 32 |
| MaxPool1 | 2×2, stride 2 | 64 × 16 × 16 |
| Conv2 | 192 filters, 3×3, padding 1 + ReLU | 192 × 16 × 16 |
| MaxPool2 | 2×2, stride 2 | 192 × 8 × 8 |
| Conv3 | 384 filters, 3×3, padding 1 + ReLU | 384 × 8 × 8 |
| Conv4 | 256 filters, 3×3, padding 1 + ReLU | 256 × 8 × 8 |
| Conv5 | 256 filters, 3×3, padding 1 + ReLU | 256 × 8 × 8 |
| MaxPool3 | 2×2, stride 2 | 256 × 4 × 4 |
| Flatten | Flatten feature maps | 4096 |
| FC1 | 4096 → 1024 + ReLU + Dropout | 1024 |
| FC2 | 1024 → 512 + ReLU + Dropout | 512 |
| FC3 | 512 → 10 | 10 |

You may use a dropout probability of 0.5 in the fully connected layers, as in the original AlexNet.

#### Implementation considerations

- Apply ReLU after each convolutional layer and the first two fully connected layers.
- Use max pooling to progressively reduce spatial resolution.
- Flatten the final convolutional feature maps before passing them to the fully connected layers.
- The final layer should output 10 raw logits, one for each CIFAR-10 class. Do not apply softmax before `nn.CrossEntropyLoss`.
- You may experiment with changing the number of filters or the size of the fully connected layers to investigate model capacity.

### ResNet

ResNet (Residual Network) was introduced by He et al. (2015). Its main innovation is the **residual connection**, also called a skip connection, which makes deeper networks easier to optimize.

Reference: [Deep Residual Learning for Image Recognition](https://arxiv.org/abs/1512.03385)

#### Residual learning

In a conventional CNN, a sequence of layers learns a transformation $H(x)$ directly.

In ResNet, the layers learn a residual transformation $F(x)$, and the original input is added back:

$$H(x)=F(x)+x$$

This creates a direct path through which information can propagate across layers.

For a basic residual block, the transformation is approximately:

**Input → Conv → BatchNorm → ReLU → Conv → BatchNorm → Add input → ReLU**

If the input and output dimensions are different, the shortcut connection must transform the input to match the output dimensions. A common solution is a $1 \times 1$ convolution, optionally with stride 2, followed by batch normalization.

#### ResNet-18 architecture (adpated)

ResNet-18 has 18 learnable layers: 17 convolutional layers and one fully connected layer. Its convolutional layers are organized into four stages, with two residual blocks per stage.

Each basic residual block contains two $3 \times 3$ convolutions.

For CIFAR-10, here is a recommended adaptation:

| Stage | Configuration | Output shape (C × H × W) |
|---|---|---|
| Input | RGB image | 3 × 32 × 32 |
| Initial Conv | 3×3, 64 filters, stride 1, padding 1 + BN + ReLU | 64 × 32 × 32 |
| Stage 1 | 2 residual blocks, 64 channels | 64 × 32 × 32 |
| Stage 2 | 2 residual blocks, 128 channels | 128 × 16 × 16 |
| Stage 3 | 2 residual blocks, 256 channels | 256 × 8 × 8 |
| Stage 4 | 2 residual blocks, 512 channels | 512 × 4 × 4 |
| Global Average Pooling | Average each feature map spatially | 512 × 1 × 1 |
| Flatten | Flatten pooled features | 512 |
| FC | 512 → 10 | 10 |

Unlike the original ImageNet ResNet-18, this adaptation uses a $3 \times 3$ initial convolution with stride 1 and removes the initial max-pooling layer. This preserves spatial information for the smaller CIFAR-10 images.

#### Basic residual block

Implement a reusable `BasicBlock` module containing:

1. A $3 \times 3$ convolution followed by batch normalization and ReLU.
2. A second $3 \times 3$ convolution followed by batch normalization.
3. A shortcut connection that adds the input to the output of the second batch normalization.
4. A final ReLU activation after the addition.

When the spatial dimensions and channel counts remain unchanged, use an identity shortcut:

$$y=\operatorname{ReLU}(F(x)+x)$$

When the dimensions differ, use a projection shortcut $W_s$:

$$y=\operatorname{ReLU}(F(x)+W_s(x))$$

For this implementation, use a $1 \times 1$ convolution and batch normalization for the projection shortcut.

#### Constructing the four stages

- **Stage 1:** Two residual blocks with 64 output channels. Both blocks use stride 1.
- **Stage 2:** Two residual blocks with 128 output channels. The first block uses stride 2 in its first convolution and shortcut to downsample; the second uses stride 1.
- **Stage 3:** Two residual blocks with 256 output channels, following the same downsampling pattern.
- **Stage 4:** Two residual blocks with 512 output channels, following the same downsampling pattern.

Finally, apply global average pooling and a fully connected classification layer.

#### Implementation considerations

- Implement the residual block as a separate PyTorch module and reuse it throughout the network.
- Ensure that the tensors being added in each residual connection have identical shapes.
- Use batch normalization after each convolution in the residual branches.
- Do not apply ReLU immediately before the residual addition; the second convolution is followed by batch normalization, then addition, then ReLU.
- Use global average pooling instead of large fully connected layers.
- The output should contain 10 raw logits for CIFAR-10 classification.

### Comparing the Architectures

After implementing both networks, compare them under the same training and evaluation settings. You need to explain your design choices by thinking about questions, including but not limited to:

- How do the architectures differ in training and validation accuracy?
- How many trainable parameters does each model contain?
- How do training time and peak GPU memory consumption compare?
- Does the deeper ResNet-18 consistently outperform AlexNet?
- How might residual connections affect optimization and gradient propagation?

You may need to modify the adapted networks to improve performances.

# Part B Assignment

Use this section to develop the main tasks and, optionally, the two challenges. Each task has a broad description followed by one consolidated answer code cell. You may design different interfaces, experiments and plots. Justify your choices and follow the accompanying assignment description for submission requirements.

## Task 1. Study and Prepare the Data
Study the full CIFAR-10 and prepare a reproducible data pipeline. Decide how to inspect the data, create data splits, normalize inputs and store image IDs. Keep the official test split separate from development. Explain the assumptions and choices that could affect later comparisons.

Use Part A as a reference rather than a required implementation. You are free to change your data organization and preprocessing, provided the choices are documented and applied consistently.

In [ ]:
# code

## Task 2. Implement and Compare the Models
Implement and train AlexNet and ResNet using PyTorch layers, starting from the architectures and CIFAR-10 adaptations discussed in A9. Explain any differences from the original designs, observe feature shapes and compare the models under a justified compute budget. Do not use pretrained weights or readily-available model constructors.

Choose your optimizer, training schedule, checkpoint rule and evaluation metrics. Justify your choices. Report numerical performance, parameter counts, training cost, number of trained samples and relevant limitations. You are free to define different model construction, training and evaluation rather than following part A.

In [ ]:
# code

## Task 3. Investigate Capacity and Regularization
Investigate how model capacity (n_channels/width, depth, etc.) and regularization influence learning and generalization. Select a feasible set of comparisons and justify what you change and hold fixed. Include L2 regularization, a condition without dropout and two dropout probabilities of your choice. Use learning curves and deterministic training/validation evaluation to distinguish underfitting, overfitting and incomplete optimization.

You may choose either architecture, modify different stages, or use another controlled experimental design.

In [ ]:
# code

## Task 4. Evaluate Robustness
Investigate whether training augmentation improves performance on clean and corrupted images. Choose suitable transformations and parameters, explain how they relate to image changes, and compare models on the same evaluation images. Separate random training views from reproducible evaluation corruptions. Discuss label ambiguity, changed error patterns and cost.

You may start with crop/flip or other corruptions. Develop the method on validation data, and freeze it for final testing.

In [ ]:
# code

## Task 5. Analyse Predictions and Adversarial Examples
Combine quantitative evaluation with examples of successful and unsuccessful predictions. Decide which metrics, visualizations and examples best explain the models' behaviour. Generate an adversarial input against a trained model, describe the perturbation, and assess the observed effect. Distinguish deliberately optimized perturbations from random corruption. Define success for the adversarial attacks. Explain class-specific errors, learning curves and representative images alongside the numerical results.

In [ ]:
# code

## Task 6. Benchmark and Discuss the Tradeoffs
Measure the costs that matter for your intended use and relate them to accuracy, robustness and the compute budget. Choose and describe an inference timing, device, precision and batch size. Discuss how training duration, randomness and hardware constrain the conclusions, and recommend a model or configuration using your measured evidence.

In [ ]:
# code

## Challenge 1. Model Compression
Investigate whether a trained model can be represented by a smaller or faster model without an unacceptable loss in performance. Choose a method, define what efficiency and acceptable accuracy change mean for your use case, and compare the result with an appropriate reference under a consistent protocol. Report compression or fine-tuning cost as well as deployment cost. A smaller parameter count, sparse mask or smaller file does not automatically imply faster inference.

In [ ]:
# code

## Challenge 2. Generalization Beyond the Development Conditions
Investigate whether your model generalizes to an unseen corruption family or a different, suitably labelled image domain. Choose a meaningful shift, explain its relationship to development data, and decide how to measure success. Discuss your findings.

In [ ]:
# code

## References
- [CIFAR-10 dataset](https://www.cs.toronto.edu/~kriz/cifar.html)
- [AlexNet original paper](https://papers.nips.cc/paper/4824-imagenet-classification-with-deep-convolutional-neural-networks)
- [ResNet original paper](https://arxiv.org/abs/1512.03385)
- [PyTorch Conv2d](https://docs.pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) and [LocalResponseNorm](https://docs.pytorch.org/docs/stable/generated/torch.nn.LocalResponseNorm.html)
- [FGSM background](https://docs.pytorch.org/tutorials/beginner/fgsm_tutorial.html)